### h082. 2. 贏家預測

#### 題目描述

有 n 個人要比賽，每個人的戰力為 S[1], S[2], ..., S[n]，而應變力為 T[1], T[2], ..., T[n]，編號為 1 到 n。

一開始將這 n 個人按照編號為 idx[1], idx[2], ..., idx[n] 的順序排成一列，並從陣列前端開始兩兩一組進行配對競賽，若該 round 有奇數人數，則最後一個烙單的人直接晉級下一 round，並不獲得戰力和應變力數值的增加。

每一場競賽的勝負判斷規則如下，假設第一個人的戰力為 a，應變力為 b，第二個人的戰力為 c，應變力為 d：

1. 若 ab >= cd 則第一個人獲勝，並且勝利方 (第一個人) 的戰力變為 a + ⌊cd/(2b)⌋，應變力變為 b + ⌊cd/(2a)⌋，失敗方 (第二個人) 的戰力變為 c + ⌊c/2⌋，應變力變為 d + ⌊d/2⌋。
2. 若 ab < cd 則第二個人獲勝，並且勝利方 (第二個人) 的戰力變為 c + ⌊ab/(2d)⌋，應變力變為 d + ⌊ab/(2c)⌋，失敗方 (第一個人) 的戰力變為 a + ⌊a/2⌋，應變力變為 b + ⌊b/2⌋。

以上除法皆為無條件捨去。

該 round 每組勝負揭曉後，按照原有順序將他們排列並分成勝利組和失敗組，失敗組中若有人已經輸了 m 次則被淘汰，再將失敗組接在勝利組之後形成新的排列進行下一 round，直到僅剩一個人成為最終勝利者，並輸出他的編號。

**輸入說明**
- 第一行輸入兩個正整數 n 和 m
- 第二行有 n 個正整數代表每個人分別的戰力值
- 第三行有 n 個正整數代表每個人分別的應變力值  
- 第四行有 n 個正整數代表第一 round 的初始排列順序

**數字範圍**
- 2 <= n <= 1000, 1 <= m <= 5
- 1 <= S[i], T[i] <= 100
- 計算過程中 ab 相乘的數值可能超過 2^32，但保證數值不超過 2^60

**子題配分**
- (50%): 2 <= n <= 100, m = 1
- (50%): 無額外限制

**輸出說明**
- 輸出最終贏家的編號

#### 測試範例

**範例輸入 #1**
```
4 1
4 2 5 3
2 5 1 5
1 2 3 4
```

**範例輸出 #1**
```
4
```

**範例輸入 #2**
```
4 5
4 1 5 3
6 5 1 6
4 1 3 2
```

**範例輸出 #2**
```
1
```

#### 程式碼解答

```python 
class Player:
    def __init__(self, id, strength, adaptability):
        self.id = id  # 玩家編號
        self.strength = strength  # 戰力值
        self.adaptability = adaptability  # 應變力值
        self.lose_count = 0  # 失敗次數
    
    def compete(self, other):
        """與另一個玩家競爭，返回勝利者和失敗者"""
        a, b = self.strength, self.adaptability
        c, d = other.strength, other.adaptability
        
        # 根據戰力與應變力乘積決定勝負
        if a * b >= c * d:
            # 自己獲勝
            self.update_winner_stats(other)  # 更新勝利者能力
            other.update_loser_stats()  # 更新失敗者能力
            other.lose_count += 1  # 增加失敗者失敗次數
            return self, other
        else:
            # 對方獲勝
            other.update_winner_stats(self)  # 更新勝利者能力
            self.update_loser_stats()  # 更新失敗者能力
            self.lose_count += 1  # 增加自己失敗次數
            return other, self
    
    def update_winner_stats(self, loser):
        """勝利後更新能力值"""
        a, b = self.strength, self.adaptability
        c, d = loser.strength, loser.adaptability
        
        # 勝利者能力更新公式
        self.strength = a + (c * d) // (2 * b)
        self.adaptability = b + (c * d) // (2 * a)
    
    def update_loser_stats(self):
        """失敗後更新能力值"""
        # 失敗者能力更新公式
        self.strength = self.strength + self.strength // 2
        self.adaptability = self.adaptability + self.adaptability // 2
    
    def __repr__(self):
        return f"Player({self.id}, 戰力:{self.strength}, 應變力:{self.adaptability}, 失敗:{self.lose_count})"


class Tournament:
    def __init__(self, n, m, strengths, adaptabilities, initial_order):
        self.n = n  # 總玩家數
        self.m = m  # 最大允許失敗次數
        self.players = self._create_players(strengths, adaptabilities)  # 建立所有玩家
        self.current_round = [self.players[i] for i in initial_order]  # 當前回合玩家順序
    
    def _create_players(self, strengths, adaptabilities):
        """從輸入資料建立玩家物件"""
        players = {}
        for i in range(1, self.n + 1):
            # 建立編號 i 的玩家，使用對應的戰力和應變力
            players[i] = Player(i, strengths[i-1], adaptabilities[i-1])
        return players
    
    def run_round(self):
        """執行一個回合的比賽"""
        winners = []  # 勝利者列表
        losers = []   # 失敗者列表
        
        # 將玩家兩兩配對進行比賽
        i = 0
        while i < len(self.current_round):
            if i + 1 >= len(self.current_round):
                # 奇數人數：最後一名玩家直接晉級
                winners.append(self.current_round[i])
                break
            
            # 取得配對的兩名玩家
            player1 = self.current_round[i]
            player2 = self.current_round[i + 1]
            
            # 進行比賽，獲得勝利者和失敗者
            winner, loser = player1.compete(player2)
            
            # 將結果加入對應列表
            winners.append(winner)
            losers.append(loser)
            
            i += 2  # 移動到下一個配對
        
        # 淘汰失敗次數達到 m 的玩家
        next_round = winners.copy()  # 勝利者全部晉級
        for player in losers:
            if player.lose_count < self.m:
                # 失敗次數未達上限的玩家可以繼續參賽
                next_round.append(player)
        
        # 更新下一回合的玩家順序
        self.current_round = next_round
    
    def run_tournament(self):
        """執行整個錦標賽直到產生最終勝利者"""
        while len(self.current_round) > 1:
            self.run_round()  # 持續進行回合直到只剩一名玩家
        
        # 返回最終勝利者的編號
        return self.current_round[0].id


def main():
    # 讀取輸入資料
    n, m = map(int, input().split())  # 玩家數和最大失敗次數
    strengths = list(map(int, input().split()))  # 戰力值列表
    adaptabilities = list(map(int, input().split()))  # 應變力值列表
    initial_order = list(map(int, input().split()))  # 初始順序
    
    # 建立並執行錦標賽
    tournament = Tournament(n, m, strengths, adaptabilities, initial_order)
    winner_id = tournament.run_tournament()
    
    # 輸出最終勝利者編號
    print(winner_id)


if __name__ == "__main__":
    main()
            
```

---

### h082. 2. Winner Prediction

#### Problem Description

There are n people in a competition. Each person has strength S[1], S[2], ..., S[n] and adaptability T[1], T[2], ..., T[n], with IDs from 1 to n.

Initially, these n people are arranged in the order idx[1], idx[2], ..., idx[n] and compete in pairs from the front of the array. If a round has an odd number of people, the last person directly advances to the next round without any increase in strength or adaptability.

The rules for determining the winner of each match are as follows. Suppose the first person has strength a and adaptability b, and the second person has strength c and adaptability d:

1. If ab >= cd, the first person wins. The winner's strength becomes a + ⌊cd/(2b)⌋, adaptability becomes b + ⌊cd/(2a)⌋. The loser's strength becomes c + ⌊c/2⌋, adaptability becomes d + ⌊d/2⌋.
2. If ab < cd, the second person wins. The winner's strength becomes c + ⌊ab/(2d)⌋, adaptability becomes d + ⌊ab/(2c)⌋. The loser's strength becomes a + ⌊a/2⌋, adaptability becomes b + ⌊b/2⌋.

All divisions use floor division.

After each round, arrange the winners and losers in their original order, separate them into winner and loser groups. Eliminate any loser who has lost m times. Then concatenate the loser group after the winner group to form the new arrangement for the next round. Continue until only one person remains as the final winner, and output their ID.

**Input Specification**
- First line: two positive integers n and m
- Second line: n positive integers representing strength values
- Third line: n positive integers representing adaptability values
- Fourth line: n positive integers representing the initial arrangement order

**Number Range**
- 2 <= n <= 1000, 1 <= m <= 5
- 1 <= S[i], T[i] <= 100
- During calculation, the product ab may exceed 2^32 but is guaranteed not to exceed 2^60

**Scoring**
- (50%): 2 <= n <= 100, m = 1
- (50%): No additional constraints

**Output Specification**
- Output the ID of the final winner

#### Test Examples

**Example Input #1**
```
4 1
4 2 5 3
2 5 1 5
1 2 3 4
```

**Example Output #1**
```
4
```

**Example Input #2**
```
4 5
4 1 5 3
6 5 1 6
4 1 3 2
```

**Example Output #2**
```
1
```

#### Code Solution

```python
class Player:
    def __init__(self, id, strength, adaptability):
        self.id = id
        self.strength = strength
        self.adaptability = adaptability
        self.lose_count = 0
    
    def compete(self, other):
        """Compete with another player and return the winner"""
        a, b = self.strength, self.adaptability
        c, d = other.strength, other.adaptability
        
        if a * b >= c * d:
            # self wins
            self.update_winner_stats(other)
            other.update_loser_stats()
            other.lose_count += 1
            return self, other
        else:
            # other wins
            other.update_winner_stats(self)
            self.update_loser_stats()
            self.lose_count += 1
            return other, self
    
    def update_winner_stats(self, loser):
        """Update stats after winning"""
        a, b = self.strength, self.adaptability
        c, d = loser.strength, loser.adaptability
        
        self.strength = a + (c * d) // (2 * b)
        self.adaptability = b + (c * d) // (2 * a)
    
    def update_loser_stats(self):
        """Update stats after losing"""
        self.strength = self.strength + self.strength // 2
        self.adaptability = self.adaptability + self.adaptability // 2
    
    def __repr__(self):
        return f"Player({self.id}, S:{self.strength}, T:{self.adaptability}, L:{self.lose_count})"


class Tournament:
    def __init__(self, n, m, strengths, adaptabilities, initial_order):
        self.n = n
        self.m = m
        self.players = self._create_players(strengths, adaptabilities)
        self.current_round = [self.players[i] for i in initial_order]
    
    def _create_players(self, strengths, adaptabilities):
        """Create Player objects from input data"""
        players = {}
        for i in range(1, self.n + 1):
            players[i] = Player(i, strengths[i-1], adaptabilities[i-1])
        return players
    
    def run_round(self):
        """Run one round of the tournament"""
        winners = []
        losers = []
        
        # Pair up players for competition
        i = 0
        while i < len(self.current_round):
            if i + 1 >= len(self.current_round):
                # Odd number: last player advances directly
                winners.append(self.current_round[i])
                break
            
            player1 = self.current_round[i]
            player2 = self.current_round[i + 1]
            
            winner, loser = player1.compete(player2)
            
            winners.append(winner)
            losers.append(loser)
            
            i += 2
        
        # Eliminate players who have lost m times
        next_round = winners.copy()
        for player in losers:
            if player.lose_count < self.m:
                next_round.append(player)
        
        self.current_round = next_round
    
    def run_tournament(self):
        """Run the entire tournament until one winner remains"""
        while len(self.current_round) > 1:
            self.run_round()
        
        return self.current_round[0].id


def main():
    # Read input
    n, m = map(int, input().split())
    strengths = list(map(int, input().split()))
    adaptabilities = list(map(int, input().split()))
    initial_order = list(map(int, input().split()))
    
    # Create and run tournament
    tournament = Tournament(n, m, strengths, adaptabilities, initial_order)
    winner_id = tournament.run_tournament()
    
    # Output final winner's ID
    print(winner_id)


if __name__ == "__main__":
    main()
```

h082. 2. 贏家預測

內容
有 n
 的人要比賽，每個人的戰力為 S[1],S[2],...S[n]
，而應變力為 T[1],T[2],...,T[n]
，編號為 1
 到 n
。

一開始將這 n
 個人按照編號為idx[1],idx[2],...,idx[n] 
 的順序排成一列，並從陣列前端開始兩兩一組進行配對競賽，若該 round 有奇數人數，則最後一個烙單的人直接晉級下一 round，並不獲得戰力和應變力數值的增加。

每一場競賽的勝負判斷規則如下，假設第一個人的戰力為 a
，應變力為 b
，第二個人的戰力為c 
，應變力為 d

1. 若 ab>=cd
 則第一個人獲勝，並且勝利方 (第一個人) 的戰力變為 a+cd/(2b)
，應變力變為 b+cd/(2a)
，失敗方 (第二個人) 的戰力變為 c+c/2
，應變力變為 d+d/2
。
2. 若 ab<cd
 則第二個人獲勝，並且勝利方 (第二個人) 的戰力變為 c+ab/(2d)
, 應變力變為 d+ab/(2c)
，失敗方 (第一個人) 的戰力變為 a+a/2
，應變力變為 b+b/2
。

以上除法皆為無條件捨去

該 round 每組勝負揭曉後，按照原有順序將他們排列並分成勝利組和失敗組，失敗組中若有人已經輸了m 
 次則被淘汰，再將失敗組接在勝利組之後形成新的排列進行下一 round，直到僅剩一個人成為最終勝利者，並輸出他的編號。

輸入說明
第一行輸入兩個正整數 n
 和 m
，接下來一行有 n
 個正整數代表每個人分別的戰力值，接下來一行有 n
 個正整數代表每個人分別的應變力值，最後一行有 n
 個正整數代表第一 round 的初始排列順序。計算過程中 ab
 相乘的數值可能超過 2^32
，但保證數值不超過 2^60
。



數字範圍

- 2<=n<=1000, 1<=m<=5
- 1<=S[i],T[i]<=100

子題配分

(50%): 2<=n<=100, m=1
(50%): 無額外限制

輸出說明
輸出最終贏家的編號。

範例輸入 #1
4 1
4 2 5 3
2 5 1 5
1 2 3 4
範例輸出 #1
4
範例輸入 #2
4 5
4 1 5 3
6 5 1 6
4 1 3 2
範例輸出 #2
1

In [ ]:
class Player:
    def __init__(self, id, S, T):
        self.id = id
        self.S=S
        self.T=T
        self.m=0
    def play(self, p):
        if self.S*self.T>=p.S*p.T:
            old_S = self.S
            self.S += p.S*p.T//(2*self.T)
            self.T += p.S*p.T//(2*old_S)
            p.S+=p.S//2
            p.T+=p.T//2
            p.m+=1
            return True
        else:
            old_S = p.S
            p.S+=self.S*self.T//(2*p.T)
            p.T+=self.S*self.T//(2*old_S)
            self.S+=self.S//2
            self.T+=self.T//2
            self.m+=1
            return False
        


# n,m = 4, 5
# Ss=[4, 1, 5, 3]
# Ts=[6, 5, 1, 6]
# Os = [4, 1, 3, 2]

# n,m = 4, 1
# Ss=[4, 2, 5, 3]
# Ts=[2, 5, 1, 5]
# Os=[1, 2, 3, 4]

n,m = map(int, input().split())
Ss = list(map(int, input().split()))
Ts = list(map(int, input().split()))
Os = list(map(int, input().split()))

Ps = [None]*n
for i in range(n):
    p = Player(i+1, Ss[i], Ts[i])
    Ps[Os[i]-1] =  p
#
r = n
win=[]
lose=[]
while r>1:
    for i in range(1, r, 2):
        if Ps[i-1].play(Ps[i]):
            win.append(Ps[i-1])
            if Ps[i].m<m:
                lose.append(Ps[i])
        else:
            if Ps[i-1].m<m:
                lose.append(Ps[i-1])
            win.append(Ps[i])
    if r%2==1:
        win.append(Ps[-1])
    Ps = win + lose
    win =[]
    lose=[]
    r = len(Ps)
print(Ps[0].id)           